# Nova real-world rectangular patch training + dataset evaluation

This notebook trains a **new patch** using the real robot preprocessing effect.

Your dataset images are already **640×640**, but the robot camera frame is **1920×1200** and is squeezed to **640×640**. Therefore, a physical square patch is not seen as a square by the model. It becomes a vertical rectangle.

For direct resize:

```text
x scale = 640 / 1920 = 0.3333
y scale = 640 / 1200 = 0.5333
y/x distortion = 0.5333 / 0.3333 = 1.6
```

So during optimization we keep the learnable patch texture square, but before overlaying it on the 640×640 dataset image we warp it to:

```text
seen_width  = base_width
seen_height = base_width × 1.6
```

Example: `80×80` learnable/printable patch → `80×128` model-seen patch.

In [ ]:
# =========================
# Install dependencies
# =========================
!pip -q install ultralytics kornia

In [ ]:
# =========================
# Imports
# =========================
import os
import random
import shutil
import json
from glob import glob
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.utils as vutils

import kornia
from ultralytics import YOLO

In [ ]:
# =========================
# Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

## 1. Main configuration

Edit these paths if your folders differ.

Expected dataset structure:

```text
nova_test_dataset_final/
  images/
  labels/
  data.yaml
  yolov8n_nova_office_best.pt
```

In [ ]:
# =========================
# Paths
# =========================
ROOT = Path('/content/drive/MyDrive/nova_test_dataset_final')
IMAGE_DIR = ROOT / 'images'
LABEL_DIR = ROOT / 'labels'
EXISTING_YAML = ROOT / 'data.yaml'
YOLO_MODEL_PATH = ROOT / 'yolov8n_nova_office_best.pt'

# Output folders
SAVE_ROOT = ROOT / 'realworld_rect_patch_training'
PATCH_SAVE_DIR = SAVE_ROOT / 'patch_outputs'
PATCH_DATASET_ROOT = SAVE_ROOT / 'patched_dataset'
EVAL_SAVE_DIR = SAVE_ROOT / 'eval_results'
VIS_SAVE_DIR = SAVE_ROOT / 'visualizations'

for p in [SAVE_ROOT, PATCH_SAVE_DIR, PATCH_DATASET_ROOT, EVAL_SAVE_DIR, VIS_SAVE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert IMAGE_DIR.exists(), f'Missing image folder: {IMAGE_DIR}'
assert LABEL_DIR.exists(), f'Missing label folder: {LABEL_DIR}'
assert EXISTING_YAML.exists(), f'Missing data.yaml: {EXISTING_YAML}'
assert YOLO_MODEL_PATH.exists(), f'Missing model: {YOLO_MODEL_PATH}'

print('ROOT:', ROOT)
print('IMAGE_DIR:', IMAGE_DIR)
print('LABEL_DIR:', LABEL_DIR)
print('YOLO_MODEL_PATH:', YOLO_MODEL_PATH)
print('SAVE_ROOT:', SAVE_ROOT)

In [ ]:
# =========================
# Real-world distortion config
# =========================
# Robot camera frame before preprocessing
CAMERA_W = 1920
CAMERA_H = 1200

# Model/dataset input size
IMGSZ = 640

# Because the camera frame is directly resized 1920x1200 -> 640x640:
X_SCALE = IMGSZ / CAMERA_W
Y_SCALE = IMGSZ / CAMERA_H
REALWORLD_Y_OVER_X = Y_SCALE / X_SCALE  # 1.6

# Learnable patch texture is square. The model-seen patch becomes rectangular.
LEARNABLE_PATCH_SIDE = 80      # saved patch texture: 80x80
BASE_SEEN_W = LEARNABLE_PATCH_SIDE
BASE_SEEN_H = int(round(LEARNABLE_PATCH_SIDE * REALWORLD_Y_OVER_X))

# Old notebook used top-left x=300, y=300 for an 80x80 square patch.
# Keep the same CENTER, but use the new rectangular height.
OLD_X_OFFSET = 300
OLD_Y_OFFSET = 300
OLD_PATCH_SIDE = 80
ANCHOR_CENTER_X = OLD_X_OFFSET + OLD_PATCH_SIDE / 2
ANCHOR_CENTER_Y = OLD_Y_OFFSET + OLD_PATCH_SIDE / 2

# Base top-left for the real-world rectangle, center-preserving.
X_OFFSET_BASE = int(round(ANCHOR_CENTER_X - BASE_SEEN_W / 2))
Y_OFFSET_BASE = int(round(ANCHOR_CENTER_Y - BASE_SEEN_H / 2))

print(f'Camera resize: {CAMERA_W}x{CAMERA_H} -> {IMGSZ}x{IMGSZ}')
print(f'X_SCALE = {X_SCALE:.4f}, Y_SCALE = {Y_SCALE:.4f}, Y/X distortion = {REALWORLD_Y_OVER_X:.4f}')
print(f'Learnable square patch: 3x{LEARNABLE_PATCH_SIDE}x{LEARNABLE_PATCH_SIDE}')
print(f'Model-seen base patch: 3x{BASE_SEEN_H}x{BASE_SEEN_W}')
print(f'Old square top-left: ({OLD_X_OFFSET}, {OLD_Y_OFFSET})')
print(f'Center-preserved rectangle top-left: ({X_OFFSET_BASE}, {Y_OFFSET_BASE})')

In [ ]:
# =========================
# Load model + class YAML
# =========================
with open(EXISTING_YAML, 'r') as f:
    existing_data = yaml.safe_load(f)

raw_names = existing_data.get('names')
if isinstance(raw_names, dict):
    names = {int(k): v for k, v in raw_names.items()}
else:
    names = {i: n for i, n in enumerate(raw_names)}

DATA_YAML_CLEAN = SAVE_ROOT / 'nova_clean_eval.yaml'
data_clean = {
    'path': str(ROOT),
    'train': 'images',
    'val': 'images',
    'test': 'images',
    'names': names,
}
DATA_YAML_CLEAN.write_text(yaml.safe_dump(data_clean, sort_keys=False))

model = YOLO(str(YOLO_MODEL_PATH))
device = next(model.model.parameters()).device
model.model.eval()

nc = int(getattr(model.model, 'nc', len(names)))

print('Classes:', names)
print('Clean eval YAML:', DATA_YAML_CLEAN)
print('Model device:', device)
print('Model nc:', nc)

In [ ]:
# =========================
# Collect dataset images
# =========================
image_paths = sorted(
    glob(str(IMAGE_DIR / '*.png')) +
    glob(str(IMAGE_DIR / '*.jpg')) +
    glob(str(IMAGE_DIR / '*.jpeg')) +
    glob(str(IMAGE_DIR / '*.webp'))
)
label_paths = sorted(glob(str(LABEL_DIR / '*.txt')))

print(f'Found {len(image_paths)} images')
print(f'Found {len(label_paths)} labels')
assert len(image_paths) > 0, 'No images found.'

# Your images should already be 640x640. This checks a few.
for p in image_paths[:10]:
    im = Image.open(p).convert('RGB')
    if im.size != (IMGSZ, IMGSZ):
        print('WARNING: not 640x640:', p, im.size)

## 2. Patch transform functions

Key idea:

1. Optimize a square patch texture.
2. Apply appearance/EOT augmentations.
3. Warp it into the real-world model-seen rectangle.
4. Overlay the rectangle on your already-640×640 dataset image.

In [ ]:
# =========================
# Image and patch transforms
# =========================
# Dataset is already 640x640, but Resize is kept as a safety fallback.
image_transform = T.Compose([
    T.ToTensor(),
    T.Resize((IMGSZ, IMGSZ), antialias=True),
])

# Appearance transforms on the square texture before real-world squeeze/stretch.
appearance_transforms = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomPerspective(distortion_scale=0.25, p=0.5),
    T.GaussianBlur(kernel_size=3),
])

def apply_appearance_transforms(patch_square: torch.Tensor) -> torch.Tensor:
    """Apply brightness/contrast/perspective/blur to square patch texture."""
    return appearance_transforms(patch_square)


def apply_eot_square(patch_square: torch.Tensor) -> torch.Tensor:
    """Apply differentiable affine EOT to the square patch texture."""
    aug = kornia.augmentation.RandomAffine(
        degrees=20,
        translate=0.08,
        scale=(0.85, 1.15),
        shear=(-5.0, 5.0),
        p=1.0,
        keepdim=True,
    ).to(patch_square.device)
    return aug(patch_square.unsqueeze(0)).squeeze(0)


def realworld_warp_to_seen_rectangle(
    patch_square: torch.Tensor,
    scale: float = 1.0,
    y_over_x: float = REALWORLD_Y_OVER_X,
    base_seen_w: int = BASE_SEEN_W,
) -> torch.Tensor:
    """
    Convert square physical/printable patch texture into the rectangle seen by the 640x640 model.

    The horizontal size follows base_seen_w * scale.
    The vertical size follows base_seen_w * y_over_x * scale.

    This is the important fix.
    """
    seen_w = max(1, int(round(base_seen_w * scale)))
    seen_h = max(1, int(round(base_seen_w * y_over_x * scale)))

    return F.interpolate(
        patch_square.unsqueeze(0),
        size=(seen_h, seen_w),
        mode='bilinear',
        align_corners=False,
    ).squeeze(0)


def center_preserving_offsets(seen_h: int, seen_w: int):
    """Return top-left x,y so each scaled rectangle keeps the same anchor center."""
    x = int(round(ANCHOR_CENTER_X - seen_w / 2))
    y = int(round(ANCHOR_CENTER_Y - seen_h / 2))
    return x, y


def overlay_patch(image: torch.Tensor, patch_seen: torch.Tensor, x_offset: int, y_offset: int) -> torch.Tensor:
    """Overlay patch_seen (C,H,W) on image (C,H,W). Offsets are clamped to valid range."""
    patched = image.clone()
    _, H_img, W_img = image.shape
    _, H_patch, W_patch = patch_seen.shape

    max_x = W_img - W_patch
    max_y = H_img - H_patch
    if max_x < 0 or max_y < 0:
        raise ValueError(f'Patch larger than image: image={H_img,W_img}, patch={H_patch,W_patch}')

    x = min(max(int(x_offset), 0), max_x)
    y = min(max(int(y_offset), 0), max_y)
    patched[:, y:y + H_patch, x:x + W_patch] = patch_seen
    return patched


def make_seen_patch_for_training(patch_square: torch.Tensor, scale: float) -> torch.Tensor:
    """Full differentiable patch pipeline used inside optimization."""
    p = apply_appearance_transforms(patch_square)
    p = apply_eot_square(p)
    p = p.clamp(0, 1)
    p = realworld_warp_to_seen_rectangle(p, scale=scale).clamp(0, 1)
    return p


def make_seen_patch_for_eval(patch_square: torch.Tensor, scale: float = 1.0) -> torch.Tensor:
    """Deterministic patch pipeline for creating patched dataset/eval images."""
    return realworld_warp_to_seen_rectangle(patch_square.clamp(0, 1), scale=scale).clamp(0, 1)

In [ ]:
# =========================
# Quick sanity preview: square patch -> rectangular seen patch
# =========================
test_patch = torch.rand(3, LEARNABLE_PATCH_SIDE, LEARNABLE_PATCH_SIDE, device=device)
test_seen = make_seen_patch_for_eval(test_patch, scale=1.0)
print('Square learnable patch:', tuple(test_patch.shape))
print('Real-world seen patch:', tuple(test_seen.shape))

# Visualize on one image
sample_img = Image.open(image_paths[0]).convert('RGB')
sample_tensor = image_transform(sample_img).to(device)
_, ph, pw = test_seen.shape
x0, y0 = center_preserving_offsets(ph, pw)
preview = overlay_patch(sample_tensor, test_seen, x0, y0)

preview_path = VIS_SAVE_DIR / 'sanity_random_rect_overlay.png'
vutils.save_image(preview.detach().cpu(), str(preview_path))
print('Saved sanity preview:', preview_path)

plt.figure(figsize=(6, 6))
plt.imshow(preview.detach().cpu().permute(1, 2, 0))
plt.axis('off')
plt.title(f'Random patch seen as {pw}x{ph} rectangle at ({x0},{y0})')
plt.show()

## 3. YOLO raw-output loss utilities

This keeps the same objective style as your notebook: suppress confident detections by minimizing top confidence values and confidence above a threshold.

In [ ]:
# =========================
# YOLO raw-output confidence parser
# =========================
def get_any_confidence(preds, nc: int) -> torch.Tensor:
    """
    Handles YOLOv8 outputs that may be (N,D), (D,N), or (1,N,D).
    D = 4 + nc OR 5 + nc.
    Returns conf_any: (N,) = max class score per prediction, multiplied by objectness if present.
    """
    if isinstance(preds, (list, tuple)):
        preds = preds[0]

    if preds.dim() == 3:
        preds = preds[0]

    # Convert (D,N) -> (N,D) if needed
    if preds.shape[0] in (4 + nc, 5 + nc) and preds.shape[1] not in (4 + nc, 5 + nc):
        preds = preds.t()

    D = preds.shape[1]
    if D == 5 + nc:
        obj = preds[:, 4]
        cls_scores = preds[:, 5:]
        return obj * cls_scores.max(dim=1).values

    if D == 4 + nc:
        cls_scores = preds[:, 4:]
        return cls_scores.max(dim=1).values

    raise ValueError(f'Unexpected preds shape {tuple(preds.shape)}. Expected D={5+nc} or D={4+nc}.')


def attack_loss_from_model_input(model, patched_input: torch.Tensor, nc: int, K: int, TH: float, LAMBDA_TH: float):
    """Forward through YOLO raw model head and compute suppression loss."""
    preds = model.model(patched_input)[0]
    conf_any = get_any_confidence(preds, nc)

    k = min(K, conf_any.numel())
    topk = torch.topk(conf_any, k=k).values

    loss_topk = -torch.log(topk + 1e-6).sum()
    loss_th = -torch.relu(conf_any - TH).sum()
    loss = loss_topk + LAMBDA_TH * loss_th

    boxes_over_th = (conf_any > TH).sum().detach().item()
    return loss, boxes_over_th

## 4. Train a new real-world-aware patch

This is the updated version of your patch loop.

Main change inside the loop:

```python
square patch -> appearance/EOT -> realworld_warp_to_seen_rectangle -> overlay -> YOLO
```

The optimizer still updates the square patch tensor. The model only ever sees the warped rectangle.

In [ ]:
# =========================
# Training config
# =========================
SEEDS = [0, 1, 2]
SEED_TAGS = {0: 'a', 1: 'b', 2: 'c'}

K = 400
TH = 0.25
LAMBDA_TH = 1.0
LR = 0.03
STEPS = 200

# Physical distance / zoom robustness.
# Each scale still uses the same y/x distortion, so square always becomes rectangle.
SCALES = [0.30, 0.40, 0.50, 0.60, 0.75, 1.00, 1.30, 1.70]

# Use None for all images each step. Use 8/16 if Colab GPU memory/time becomes annoying.
IMAGES_PER_STEP = None

# Save examples every N steps for sanity.
SAVE_PREVIEW_EVERY = 50

print('Training square patch side:', LEARNABLE_PATCH_SIDE)
print('Base seen rectangle H,W:', BASE_SEEN_H, BASE_SEEN_W)
print('Scales:', SCALES)
print('IMAGES_PER_STEP:', IMAGES_PER_STEP)

In [ ]:
# ============================================================
# FIX: GPU-safe real-world patch warp
# Replace Kornia-based make_seen_patch_for_training/eval
# ============================================================

import torch
import torch.nn.functional as F

# Disable Kornia affine EOT for now because it is causing CPU/GPU mismatch.
# The important real-world effect is still preserved:
# square printable patch -> rectangular model-seen patch.
USE_AFFINE_EOT = False


def make_seen_patch_for_training(patch_square, scale=1.0):
    """
    Takes the learnable square printable patch and warps it into the
    rectangle that the model sees after 1920x1200 -> 640x640 distortion.

    Input:
        patch_square: Tensor [3, H, W] on CUDA

    Output:
        seen_patch: Tensor [3, BASE_SEEN_H*scale, BASE_SEEN_W*scale] on same device
    """

    patch_square = patch_square.to(device)

    target_w = max(1, int(round(BASE_SEEN_W * scale)))
    target_h = max(1, int(round(BASE_SEEN_H * scale)))

    seen_patch = F.interpolate(
        patch_square.unsqueeze(0),
        size=(target_h, target_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)

    return seen_patch.clamp(0, 1)


def make_seen_patch_for_eval(patch_square, scale=1.0):
    """
    Same as training version, but for evaluation/preview.
    """

    patch_square = patch_square.to(device)

    target_w = max(1, int(round(BASE_SEEN_W * scale)))
    target_h = max(1, int(round(BASE_SEEN_H * scale)))

    seen_patch = F.interpolate(
        patch_square.unsqueeze(0),
        size=(target_h, target_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)

    return seen_patch.clamp(0, 1)


# Quick sanity test
_test_patch = torch.rand(
    3,
    LEARNABLE_PATCH_SIDE,
    LEARNABLE_PATCH_SIDE,
    device=device,
)

_test_seen = make_seen_patch_for_training(_test_patch, scale=1.0)

print("Patch square shape:", tuple(_test_patch.shape), "| device:", _test_patch.device)
print("Seen patch shape:", tuple(_test_seen.shape), "| device:", _test_seen.device)
print("Expected seen size:", f"{BASE_SEEN_W}x{BASE_SEEN_H}")

In [ ]:
# =========================
# GPU-fixed real-world-aware training loop
# Paste this over your current loop cell
# =========================

import time
import random
import json
import numpy as np
import pandas as pd
import torch
import torchvision.utils as vutils
from PIL import Image

# -------------------------
# Force GPU setup
# -------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("Using torch device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Move YOLO model to GPU and freeze it
model.to(device)
model.model.to(device)
model.model.eval()

for p in model.model.parameters():
    p.requires_grad_(False)

print("YOLO model device:", next(model.model.parameters()).device)

# -------------------------
# Safety settings
# -------------------------
# Your old IMAGES_PER_STEP=None means "use the whole dataset per step".
# That is extremely slow. This forces a reasonable mini-batch.
if IMAGES_PER_STEP is None:
    EFFECTIVE_IMAGES_PER_STEP = min(8, len(image_paths))
else:
    EFFECTIVE_IMAGES_PER_STEP = min(IMAGES_PER_STEP, len(image_paths))

print("Images per optimization step:", EFFECTIVE_IMAGES_PER_STEP)
print("Scales:", SCALES)
print("Steps:", STEPS)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Faster on GPU
    torch.backends.cudnn.benchmark = True


seed_results = {}
all_logs = []

for seed in SEEDS:
    tag = SEED_TAGS.get(seed, str(seed))
    save_path = PATCH_SAVE_DIR / f"realworld_rect_patch_seed{seed}_{tag}.pt"

    print("\n" + "=" * 70)
    print(f"Running seed = {seed} -> {save_path}")
    print("=" * 70)

    set_seed(seed)

    # This is the square printable/learnable patch.
    patch_square = torch.rand(
        3,
        LEARNABLE_PATCH_SIDE,
        LEARNABLE_PATCH_SIDE,
        device=device,
        requires_grad=True,
    )

    optimizer = torch.optim.Adam([patch_square], lr=LR)

    print("Patch device:", patch_square.device)

    best_loss = float("inf")
    best_patch_cpu = patch_square.detach().clone().cpu()

    for step in range(STEPS):
        step_start = time.time()

        optimizer.zero_grad(set_to_none=True)

        # Use mini-batch instead of full dataset
        step_paths = random.sample(
            image_paths,
            k=EFFECTIVE_IMAGES_PER_STEP,
        )

        denom = len(step_paths) * len(SCALES)

        total_loss_val = 0.0
        total_boxes_over_th = 0

        # Accumulate losses, then do ONE backward per step
        step_loss = 0.0

        for img_i, img_path in enumerate(step_paths):
            img = Image.open(img_path).convert("RGB")
            image_tensor = image_transform(img).to(device, non_blocking=True)

            for sc in SCALES:
                # Square printable patch -> real-world seen rectangle
                seen_patch = make_seen_patch_for_training(patch_square, scale=sc)
                seen_patch = seen_patch.to(device)

                _, ph, pw = seen_patch.shape
                x_off, y_off = center_preserving_offsets(ph, pw)

                patched_image = overlay_patch(
                    image_tensor,
                    seen_patch,
                    x_off,
                    y_off,
                )

                patched_input = patched_image.unsqueeze(0).to(device)

                loss, boxes_over_th = attack_loss_from_model_input(
                    model=model,
                    patched_input=patched_input,
                    nc=nc,
                    K=K,
                    TH=TH,
                    LAMBDA_TH=LAMBDA_TH,
                )

                step_loss = step_loss + loss / max(1, denom)

                total_loss_val += float(loss.detach().cpu().item())
                total_boxes_over_th += boxes_over_th

        step_loss.backward()
        optimizer.step()

        with torch.no_grad():
            patch_square.clamp_(0, 1)

        avg_loss = total_loss_val / max(1, denom)
        avg_boxes_over_th = total_boxes_over_th / max(1, denom)
        step_time = time.time() - step_start

        all_logs.append({
            "seed": seed,
            "step": step,
            "avg_loss": avg_loss,
            "avg_boxes_over_th": avg_boxes_over_th,
            "base_seen_h": BASE_SEEN_H,
            "base_seen_w": BASE_SEEN_W,
            "y_over_x": REALWORLD_Y_OVER_X,
            "step_time_sec": step_time,
            "device": str(device),
            "images_per_step": EFFECTIVE_IMAGES_PER_STEP,
            "num_scales": len(SCALES),
        })

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_patch_cpu = patch_square.detach().clone().cpu()
            torch.save(best_patch_cpu, save_path)

        if step % 1 == 0:
            gpu_mem = ""
            if torch.cuda.is_available():
                gpu_mem = f" | GPU mem: {torch.cuda.memory_allocated() / 1024**2:.1f} MB"

            print(
                f"[seed {seed}] Step {step:3d}/{STEPS} | "
                f"Loss: {avg_loss:.4f} | "
                f"Avg boxes>TH({TH}): {avg_boxes_over_th:.2f} | "
                f"seen={BASE_SEEN_W}x{BASE_SEEN_H} | "
                f"time={step_time:.2f}s"
                f"{gpu_mem}"
            )

        if SAVE_PREVIEW_EVERY and step % SAVE_PREVIEW_EVERY == 0:
            with torch.no_grad():
                seen = make_seen_patch_for_eval(patch_square.detach(), scale=1.0).to(device)
                _, ph, pw = seen.shape
                x_off, y_off = center_preserving_offsets(ph, pw)

                preview = overlay_patch(
                    image_tensor,
                    seen,
                    x_off,
                    y_off,
                )

                vutils.save_image(
                    patch_square.detach().cpu(),
                    str(VIS_SAVE_DIR / f"seed{seed}_step{step:03d}_square_patch.png"),
                )

                vutils.save_image(
                    seen.detach().cpu(),
                    str(VIS_SAVE_DIR / f"seed{seed}_step{step:03d}_seen_rect_patch.png"),
                )

                vutils.save_image(
                    preview.detach().cpu(),
                    str(VIS_SAVE_DIR / f"seed{seed}_step{step:03d}_overlay_preview.png"),
                )

    # Final save for this seed
    torch.save(best_patch_cpu, save_path)

    print(f"Saved seed {seed} best square patch to: {save_path}")
    print(f"Best loss for seed {seed}: {best_loss:.6f}")

    seed_results[seed] = {
        "best_loss": best_loss,
        "best_patch_cpu": best_patch_cpu,
        "save_path": str(save_path),
    }


# -------------------------
# Choose overall best patch
# -------------------------
best_seed = min(seed_results.keys(), key=lambda s: seed_results[s]["best_loss"])
best_patch_overall = seed_results[best_seed]["best_patch_cpu"]

best_path = PATCH_SAVE_DIR / "realworld_rect_patch_best.pt"
torch.save(best_patch_overall, best_path)

# -------------------------
# Save logs
# -------------------------
logs_df = pd.DataFrame(all_logs)
logs_csv = PATCH_SAVE_DIR / "training_log.csv"
logs_df.to_csv(logs_csv, index=False)

summary_json = PATCH_SAVE_DIR / "seed_summary.json"
with open(summary_json, "w") as f:
    json.dump(
        {
            int(seed): {
                "best_loss": float(res["best_loss"]),
                "save_path": res["save_path"],
            }
            for seed, res in seed_results.items()
        },
        f,
        indent=2,
    )

print("\n" + "#" * 70)
print(f"Best seed = {best_seed} with best_loss = {seed_results[best_seed]['best_loss']:.6f}")
print(f"Saved overall best square patch to: {best_path}")
print(f"Saved training log to: {logs_csv}")
print(f"Saved summary to: {summary_json}")
print("#" * 70)

## 5. Save square patch and real-world-seen rectangle previews

The `.pt` file is the square patch texture. The `seen_rect` image shows how that square patch is warped before YOLO sees it.

In [ ]:
# =========================
# Save final patch PNG/PT aliases
# =========================
PATCH_ALIAS_PT = PATCH_SAVE_DIR / 'nova_patch_realworld_square.pt'
PATCH_ALIAS_PNG = PATCH_SAVE_DIR / 'nova_patch_realworld_square.png'
SEEN_ALIAS_PNG = PATCH_SAVE_DIR / 'nova_patch_realworld_seen_rect.png'

best_patch = torch.load(best_path, map_location='cpu').float().clamp(0, 1)
torch.save(best_patch, PATCH_ALIAS_PT)
vutils.save_image(best_patch, str(PATCH_ALIAS_PNG))

seen_rect = make_seen_patch_for_eval(best_patch.to(device), scale=1.0).detach().cpu()
vutils.save_image(seen_rect, str(SEEN_ALIAS_PNG))

print('Saved square patch PT:', PATCH_ALIAS_PT)
print('Saved square patch PNG:', PATCH_ALIAS_PNG)
print('Saved model-seen rectangular PNG:', SEEN_ALIAS_PNG)
print('Square patch shape:', tuple(best_patch.shape))
print('Seen rectangle shape:', tuple(seen_rect.shape))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(best_patch.permute(1, 2, 0))
plt.axis('off')
plt.title(f'Saved square patch {best_patch.shape[2]}x{best_patch.shape[1]}')

plt.subplot(1, 2, 2)
plt.imshow(seen_rect.permute(1, 2, 0))
plt.axis('off')
plt.title(f'Model-seen rect {seen_rect.shape[2]}x{seen_rect.shape[1]}')
plt.show()

## 6. Create a patched dataset using the trained patch

This creates a new dataset folder with patched images and copied labels. Labels are copied unchanged because the patch does not alter ground-truth object boxes.

In [ ]:
# =========================
# Build patched dataset(b) Detector-visible

PATCH_DATASET_IMAGES = PATCH_DATASET_ROOT / 'images'
PATCH_DATASET_LABELS = PATCH_DATASET_ROOT / 'labels'
PATCH_DATASET_IMAGES.mkdir(parents=True, exist_ok=True)
PATCH_DATASET_LABELS.mkdir(parents=True, exist_ok=True)

# Clear old patched images/labels to avoid mixing runs.
for folder in [PATCH_DATASET_IMAGES, PATCH_DATASET_LABELS]:
    for fp in folder.glob('*'):
        if fp.is_file():
            fp.unlink()

patch_square = torch.load(PATCH_ALIAS_PT, map_location='cpu').float().clamp(0, 1).to(device)
patch_seen = make_seen_patch_for_eval(patch_square, scale=1.0)
_, ph, pw = patch_seen.shape
x_off, y_off = center_preserving_offsets(ph, pw)

print('Using square patch:', tuple(patch_square.shape))
print('Overlaying seen rectangle:', tuple(patch_seen.shape))
print('Top-left:', (x_off, y_off))

rows = []
for idx, img_path in enumerate(image_paths):
    img_path = Path(img_path)
    img = Image.open(img_path).convert('RGB')
    img_t = image_transform(img).to(device)

    patched_t = overlay_patch(img_t, patch_seen, x_off, y_off).detach().cpu().clamp(0, 1)

    out_img_path = PATCH_DATASET_IMAGES / img_path.name
    vutils.save_image(patched_t, str(out_img_path))

    label_path = LABEL_DIR / f'{img_path.stem}.txt'
    out_label_path = PATCH_DATASET_LABELS / f'{img_path.stem}.txt'
    if label_path.exists():
        shutil.copy2(label_path, out_label_path)
    else:
        out_label_path.write_text('')

    rows.append({
        'image': img_path.name,
        'patched_image': str(out_img_path),
        'label_exists': label_path.exists(),
        'patch_x': x_off,
        'patch_y': y_off,
        'patch_w_seen': pw,
        'patch_h_seen': ph,
    })

    if idx < 5:
        print(f'[{idx}] saved {out_img_path.name}')

manifest_df = pd.DataFrame(rows)
manifest_csv = PATCH_DATASET_ROOT / 'patched_manifest.csv'
manifest_df.to_csv(manifest_csv, index=False)

# Create YOLO yaml for patched dataset
DATA_YAML_PATCHED = PATCH_DATASET_ROOT / 'data.yaml'
data_patched = {
    'path': str(PATCH_DATASET_ROOT),
    'train': 'images',
    'val': 'images',
    'test': 'images',
    'names': names,
}
DATA_YAML_PATCHED.write_text(yaml.safe_dump(data_patched, sort_keys=False))

print('\nPatched dataset created:')
print('Images:', PATCH_DATASET_IMAGES)
print('Labels:', PATCH_DATASET_LABELS)
print('YAML:', DATA_YAML_PATCHED)
print('Manifest:', manifest_csv)
print('Number of patched images:', len(list(PATCH_DATASET_IMAGES.glob('*'))))

In [ ]:
# =========================
# Preview patched dataset samples
# =========================
sample_outputs = sorted(list(PATCH_DATASET_IMAGES.glob('*')))[:6]
plt.figure(figsize=(14, 8))
for i, p in enumerate(sample_outputs):
    im = Image.open(p).convert('RGB')
    plt.subplot(2, 3, i + 1)
    plt.imshow(im)
    plt.axis('off')
    plt.title(p.name)
plt.tight_layout()
plt.show()

## 7. Evaluate clean vs patched datasets

This runs the same YOLO model on the clean dataset and on the patched dataset.

In [ ]:
%%writefile /content/evaluation_ab.py
import argparse, json, re, subprocess, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ----------------- utils -----------------
def run(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
        rc = p.wait()
    return rc


def percentile(arr, p):
    if arr is None or len(arr) == 0:
        return None
    return float(np.percentile(np.asarray(arr, dtype=np.float32), p))


def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


def model_names_as_list(model: YOLO):
    names = getattr(model, "names", None)
    if names is None:
        raise ValueError("Model has no .names; cannot build dataset YAML.")

    if isinstance(names, dict):
        max_k = max(int(k) for k in names.keys())
        out = [None] * (max_k + 1)
        for k, v in names.items():
            out[int(k)] = str(v)
        for i, v in enumerate(out):
            if v is None:
                out[i] = f"class_{i}"
        return out

    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]

    raise ValueError(f"Unsupported model.names type: {type(names)}")


def read_dataset_yaml(yaml_path: Path):
    txt = yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    d = {"path": None, "train": None, "val": None, "test": None}
    for ln in txt:
        ln = ln.strip()
        if not ln or ln.startswith("#"):
            continue
        for k in ("path", "train", "val", "test"):
            if ln.startswith(k + ":"):
                d[k] = ln.split(":", 1)[1].strip()
    return d


def _has_images(dir_path: Path) -> bool:
    if not dir_path.exists() or not dir_path.is_dir():
        return False
    for x in dir_path.iterdir():
        if x.is_file() and x.suffix.lower() in IMG_EXTS:
            return True
    return False


def get_image_dir_from_yaml(yaml_path: Path) -> Path:
    y = read_dataset_yaml(yaml_path)
    root = y.get("path")
    if not root:
        raise ValueError(f"Could not parse 'path:' from {yaml_path}")

    rel = y.get("test") or y.get("val") or y.get("train")
    if not rel:
        raise ValueError(f"Could not parse test/val/train from {yaml_path}")

    base = (Path(root) / rel).resolve()

    if _has_images(base):
        return base

    for sub in ("test", "val", "train"):
        cand = base / sub
        if _has_images(cand):
            return cand

    if base.name != "images":
        cand = (Path(root) / "images").resolve()
        if _has_images(cand):
            return cand
        for sub in ("test", "val", "train"):
            cand2 = cand / sub
            if _has_images(cand2):
                return cand2

    return base


def get_labels_dir_for_images(img_dir: Path) -> Path:
    parts = list(img_dir.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts)
    return img_dir.parent / "labels"


def write_yaml(out_yaml: Path, root: Path, test_rel: str, names):
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    content = (
        f"path: {root}\n"
        f"train: {test_rel}\n"
        f"val: {test_rel}\n"
        f"test: {test_rel}\n"
        f"\nnc: {len(names)}\n"
        f"names: {json.dumps(names)}\n"
    )
    out_yaml.write_text(content, encoding="utf-8")


def parse_yolo_val_metrics_from_log(log_path: Path):
    pat_all = re.compile(
        r"^\s*all\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$"
    )
    per_class = {}
    pat_cls = re.compile(
        r"^\s*([A-Za-z0-9 _-]+)\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$"
    )

    images = instances = None
    P = R = mAP50 = mAP5095 = None

    for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = pat_all.match(ln)
        if m:
            images = int(m.group(1))
            instances = int(m.group(2))
            P = float(m.group(3))
            R = float(m.group(4))
            mAP50 = float(m.group(5))
            mAP5095 = float(m.group(6))

        mc = pat_cls.match(ln)
        if mc:
            cls = mc.group(1).strip()
            if cls != "all":
                per_class[cls] = {
                    "images": int(mc.group(2)),
                    "instances": int(mc.group(3)),
                    "precision": float(mc.group(4)),
                    "recall": float(mc.group(5)),
                    "mAP50": float(mc.group(6)),
                    "mAP50-95": float(mc.group(7)),
                }

    return {
        "images": images,
        "instances": instances,
        "precision": P,
        "recall": R,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "per_class": per_class,
        "log_path": str(log_path),
    }


# ----------------- A) detection density -----------------
def compute_detection_density_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, det_k_list, max_det: int):
    img_dir = get_image_dir_from_yaml(yaml_path)
    names = model_names_as_list(model)

    det_counts = []
    max_conf_any = []
    has_ge_k = {str(k): 0 for k in det_k_list}
    has_any_conf_ge = {str(t): 0 for t in thresholds}
    total_dets_conf_ge = {str(t): 0 for t in thresholds}
    dets_per_img_conf_ge = {str(t): [] for t in thresholds}
    class_hist = {}

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
        max_det=max_det,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(n)

        if n == 0:
            max_conf_any.append(0.0)
            for t in thresholds:
                dets_per_img_conf_ge[str(t)].append(0)
        else:
            confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
            max_conf_any.append(float(confs.max()))
            clss = boxes.cls.detach().cpu().numpy().astype(int)

            for c in clss:
                k = names[c] if (0 <= c < len(names)) else str(int(c))
                class_hist[k] = class_hist.get(k, 0) + 1

            for t in thresholds:
                n_ge = int((confs >= t).sum())
                total_dets_conf_ge[str(t)] += n_ge
                dets_per_img_conf_ge[str(t)].append(n_ge)

        for k in det_k_list:
            if n >= k:
                has_ge_k[str(k)] += 1

        for t in thresholds:
            if max_conf_any[-1] >= t:
                has_any_conf_ge[str(t)] += 1

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[A] {yaml_path.name}: processed {n_images} images", flush=True)

    det_counts_np = np.asarray(det_counts, dtype=np.float32)
    max_conf_np = np.asarray(max_conf_any, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "detections_total_model": int(det_counts_np.sum()),
        "detections_per_image_mean": float(det_counts_np.mean()) if n_images else None,
        "detections_per_image_median": float(np.median(det_counts_np)) if n_images else None,
        "detections_per_image_p95": percentile(det_counts_np, 95),
        "detections_per_image_conf_ge_mean": {
            str(t): float(np.mean(dets_per_img_conf_ge[str(t)])) if n_images else None for t in thresholds
        },
        "max_conf_any_mean": float(max_conf_np.mean()) if n_images else None,
        "max_conf_any_median": float(np.median(max_conf_np)) if n_images else None,
        "max_conf_any_p95": percentile(max_conf_np, 95),
        "image_rate_dets_ge": {k: (v / n_images if n_images else None) for k, v in has_ge_k.items()},
        "image_rate_any_det_conf_ge": {k: (v / n_images if n_images else None) for k, v in has_any_conf_ge.items()},
        "total_detections_conf_ge": total_dets_conf_ge,
        "predicted_class_histogram": dict(sorted(class_hist.items(), key=lambda kv: kv[1], reverse=True)),
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(get_image_dir_from_yaml(yaml_path)),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "det_k_list": det_k_list,
            "max_det": max_det,
        },
    }


# ----------------- B) false positives vs GT -----------------
def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out


def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_false_positive_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, gt_iou_match: float, max_det: int):
    img_dir = get_image_dir_from_yaml(yaml_path)
    lab_dir = get_labels_dir_for_images(img_dir)

    fp_per_image = []
    fp_by_thr = {str(t): [] for t in thresholds}

    total_fp = 0
    total_tp = 0
    total_preds = 0
    total_gt = 0

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
        max_det=max_det,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        img_name = Path(r.path).name
        stem = Path(img_name).stem
        gt_path = lab_dir / f"{stem}.txt"

        gt = read_yolo_gt_labels(gt_path)
        total_gt += len(gt)

        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            fp_per_image.append(0)
            for t in thresholds:
                fp_by_thr[str(t)].append(0)
            if n_images == 1 or (n_images % 10 == 0):
                print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)
            continue

        pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
        pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

        preds = []
        for i in range(len(pred_cls)):
            c = int(pred_cls[i])
            x, y, w, h = pred_xywhn[i].tolist()
            preds.append((c, xywhn_to_xyxy(x, y, w, h), float(pred_conf[i])))

        total_preds += len(preds)

        gt_by_class = {}
        for (c, x, y, w, h) in gt:
            gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

        def count_fp_at(thr):
            pf = [(c, box, conf) for (c, box, conf) in preds if conf >= thr]
            pf.sort(key=lambda z: z[2], reverse=True)

            tp = 0
            fp = 0
            remaining = {c: list(lst) for c, lst in gt_by_class.items()}

            for c, pbox, _ in pf:
                best_iou = 0.0
                best_j = None
                gtl = remaining.get(c, [])
                for j, gbox in enumerate(gtl):
                    iouv = iou_xyxy(pbox, gbox)
                    if iouv > best_iou:
                        best_iou = iouv
                        best_j = j
                if best_j is not None and best_iou >= gt_iou_match:
                    tp += 1
                    gtl.pop(best_j)
                    remaining[c] = gtl
                else:
                    fp += 1
            return tp, fp, len(pf)

        tp0, fp0, _ = count_fp_at(conf_min)
        total_tp += tp0
        total_fp += fp0
        fp_per_image.append(fp0)

        for t in thresholds:
            _, fpt, _ = count_fp_at(t)
            fp_by_thr[str(t)].append(fpt)

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)

    fp_arr = np.asarray(fp_per_image, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "gt_total_boxes": int(total_gt),
        "pred_total_boxes_conf_ge_conf_min": int(total_preds),
        "tp_total_conf_ge_conf_min": int(total_tp),
        "fp_total_conf_ge_conf_min": int(total_fp),
        "fp_per_image_mean_conf_min": float(fp_arr.mean()) if n_images else None,
        "fp_per_image_median_conf_min": float(np.median(fp_arr)) if n_images else None,
        "fp_per_image_p95_conf_min": percentile(fp_arr, 95),
        "fp_per_image_conf_ge": {str(t): float(np.mean(fp_by_thr[str(t)])) if n_images else None for t in thresholds},
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(img_dir),
            "labels_dir_used": str(lab_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "gt_iou_match": gt_iou_match,
            "max_det": max_det,
        },
    }


def delta_numeric(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return b - a
    return None


def dict_delta(a, b):
    keys = sorted(set(a.keys()) | set(b.keys()), key=lambda x: float(x))
    return {k: delta_numeric(a.get(k), b.get(k)) for k in keys}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_root", required=True)
    ap.add_argument("--run_name", required=True)
    ap.add_argument("--model", required=True)

    ap.add_argument("--clean_root", required=True)
    ap.add_argument("--patched_root", required=True)

    ap.add_argument("--split", default="test")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--conf_min", type=float, default=0.001)
    ap.add_argument("--iou", type=float, default=0.6)
    ap.add_argument("--thresholds", default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match", type=float, default=0.5)
    ap.add_argument("--det_k", default="1,5,10,20,50")
    ap.add_argument("--max_det", type=int, default=1000)
    args = ap.parse_args()

    thresholds = [float(x.strip()) for x in args.thresholds.split(",") if x.strip()]
    det_k_list = [int(x.strip()) for x in args.det_k.split(",") if x.strip()]

    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    run_dir = out_root / args.run_name
    safe_mkdir(run_dir)

    yamls_dir = safe_mkdir(run_dir / "yamls")

    print(f"RUN DIR: {run_dir}", flush=True)
    print(f"MODEL: {args.model}", flush=True)
    print(f"IMG_SIZE: {args.imgsz}", flush=True)
    print(f"SPLIT: {args.split}", flush=True)
    print(f"MAX_DET: {args.max_det}", flush=True)

    model = YOLO(args.model)
    names = model_names_as_list(model)

    clean_img_dir = Path(args.clean_root) / "images" / args.split
    clean_lab_dir = Path(args.clean_root) / "labels" / args.split
    patched_img_dir = Path(args.patched_root) / "images" / args.split
    patched_lab_dir = Path(args.patched_root) / "labels" / args.split

    if not clean_img_dir.exists():
        raise FileNotFoundError(f"Missing clean images: {clean_img_dir}")
    if not clean_lab_dir.exists():
        raise FileNotFoundError(f"Missing clean labels: {clean_lab_dir}")
    if not patched_img_dir.exists():
        raise FileNotFoundError(f"Missing patched images: {patched_img_dir}")
    if not patched_lab_dir.exists():
        raise FileNotFoundError(f"Missing patched labels: {patched_lab_dir}")

    clean_yaml = yamls_dir / "clean.yaml"
    patched_yaml = yamls_dir / "patched.yaml"
    write_yaml(clean_yaml, Path(args.clean_root), f"images/{args.split}", names)
    write_yaml(patched_yaml, Path(args.patched_root), f"images/{args.split}", names)

    clean_val_log = run_dir / "clean_val.log"
    patched_val_log = run_dir / "patched_val.log"

    cmd_base = [
        "yolo", "val",
        f"model={args.model}",
        f"imgsz={args.imgsz}",
        f"conf={args.conf_min}",
        f"iou={args.iou}",
        f"max_det={args.max_det}",
        "split=test",
        "batch=1",
        "workers=0",
        f"project={run_dir}",
    ]

    rc = run(cmd_base + [f"data={clean_yaml}", "name=clean"], clean_val_log)
    if rc != 0:
        raise SystemExit(f"yolo val clean failed (rc={rc}) see {clean_val_log}")

    rc = run(cmd_base + [f"data={patched_yaml}", "name=patched"], patched_val_log)
    if rc != 0:
        raise SystemExit(f"yolo val patched failed (rc={rc}) see {patched_val_log}")

    clean_val = parse_yolo_val_metrics_from_log(clean_val_log)
    patched_val = parse_yolo_val_metrics_from_log(patched_val_log)

    clean_A = compute_detection_density_metrics(
        model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, args.max_det
    )
    patched_A = compute_detection_density_metrics(
        model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, args.max_det
    )

    clean_B = compute_false_positive_metrics(
        model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, args.max_det
    )
    patched_B = compute_false_positive_metrics(
        model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, args.max_det
    )

    delta = {
        "yolo_val": {
            "precision": delta_numeric(clean_val.get("precision"), patched_val.get("precision")),
            "recall": delta_numeric(clean_val.get("recall"), patched_val.get("recall")),
            "mAP50": delta_numeric(clean_val.get("mAP50"), patched_val.get("mAP50")),
            "mAP50-95": delta_numeric(clean_val.get("mAP50-95"), patched_val.get("mAP50-95")),
        },
        "A": {
            "detections_per_image_mean": delta_numeric(clean_A.get("detections_per_image_mean"), patched_A.get("detections_per_image_mean")),
            "detections_total_model": delta_numeric(clean_A.get("detections_total_model"), patched_A.get("detections_total_model")),
            "detections_per_image_conf_ge_mean": dict_delta(
                clean_A.get("detections_per_image_conf_ge_mean", {}),
                patched_A.get("detections_per_image_conf_ge_mean", {})
            ),
        },
        "B": {
            "fp_per_image_mean_conf_min": delta_numeric(clean_B.get("fp_per_image_mean_conf_min"), patched_B.get("fp_per_image_mean_conf_min")),
            "fp_total_conf_ge_conf_min": delta_numeric(clean_B.get("fp_total_conf_ge_conf_min"), patched_B.get("fp_total_conf_ge_conf_min")),
            "fp_per_image_conf_ge": dict_delta(
                clean_B.get("fp_per_image_conf_ge", {}),
                patched_B.get("fp_per_image_conf_ge", {})
            ),
        },
    }

    combined = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "max_det": args.max_det,
        "clean": {"yaml": str(clean_yaml), "yolo_val": clean_val, "A": clean_A, "B": clean_B},
        "patched": {"yaml": str(patched_yaml), "yolo_val": patched_val, "A": patched_A, "B": patched_B},
        "delta_patched_minus_clean": delta,
    }

    summary = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "max_det": args.max_det,
        "clean_mAP50": clean_val.get("mAP50"),
        "patched_mAP50": patched_val.get("mAP50"),
        "delta_mAP50": delta["yolo_val"]["mAP50"],
        "clean_mAP50_95": clean_val.get("mAP50-95"),
        "patched_mAP50_95": patched_val.get("mAP50-95"),
        "delta_mAP50_95": delta["yolo_val"]["mAP50-95"],
        "clean_det_mean": clean_A.get("detections_per_image_mean"),
        "patched_det_mean": patched_A.get("detections_per_image_mean"),
        "delta_det_mean": delta["A"]["detections_per_image_mean"],
        "clean_fp_mean": clean_B.get("fp_per_image_mean_conf_min"),
        "patched_fp_mean": patched_B.get("fp_per_image_mean_conf_min"),
        "delta_fp_mean": delta["B"]["fp_per_image_mean_conf_min"],
        "clean_det_per_img_conf_ge": clean_A.get("detections_per_image_conf_ge_mean", {}),
        "patched_det_per_img_conf_ge": patched_A.get("detections_per_image_conf_ge_mean", {}),
        "delta_det_per_img_conf_ge": delta["A"]["detections_per_image_conf_ge_mean"],
        "clean_fp_per_img_conf_ge": clean_B.get("fp_per_image_conf_ge", {}),
        "patched_fp_per_img_conf_ge": patched_B.get("fp_per_image_conf_ge", {}),
        "delta_fp_per_img_conf_ge": delta["B"]["fp_per_image_conf_ge"],
    }

    out_combined = run_dir / "combined_metrics.json"
    out_summary = run_dir / "summary.json"
    out_combined.write_text(json.dumps(combined, indent=2), encoding="utf-8")
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nWROTE:", out_combined, flush=True)
    print("WROTE:", out_summary, flush=True)
    print("\nSUMMARY:\n", json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

EVAL_SCRIPT = Path("/content/evaluation_ab.py")

txt = EVAL_SCRIPT.read_text()

# Make model.predict(...) use GPU instead of CPU
txt = txt.replace('device="cpu",', 'device="0",')

# Make YOLO CLI val use GPU too
txt = txt.replace(
    'f"max_det={args.max_det}",\n        "split=test",',
    'f"max_det={args.max_det}",\n        "device=0",\n        "split=test",'
)

EVAL_SCRIPT.write_text(txt)

print("Patched evaluation_ab.py to use GPU device=0")

In [ ]:
# =========================
# RUN evaluation_ab.py ON NOVA CLEAN vs REAL-WORLD RECT PATCHED DATASET
# =========================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

# -------------------------
# Paths
# -------------------------
EVAL_SCRIPT = Path("/content/evaluation_ab.py")

MODEL_PATH = Path("/content/drive/MyDrive/nova_test_dataset_final/yolov8n_nova_office_best.pt")

# Original clean Nova dataset.
# This must contain:
#   images/
#   labels/
CLEAN_FLAT_ROOT = Path("/content/drive/MyDrive/nova_test_dataset_final")

# Real-world rectangular patched dataset.
# This should contain:
#   images/
#   labels/
PATCHED_FLAT_ROOT = Path(
    "/content/drive/MyDrive/nova_test_dataset_final/realworld_rect_patch_training/patched_dataset"
)

# evaluation_ab.py expects:
#   root/images/test
#   root/labels/test
CLEAN_EVAL_ROOT = Path("/content/drive/MyDrive/nova_test_dataset_final/eval_format_clean")
PATCHED_EVAL_ROOT = Path("/content/drive/MyDrive/nova_test_dataset_final/eval_format_realworld_rect_patched")

SPLIT = "test"

OUT_ROOT = Path("/content/drive/MyDrive/nova_test_dataset_final/eval_ab_runs")
RUN_NAME = "nova_clean_vs_realworld_rect_patched_test"

IMGSZ = 640

# Important:
# conf_min=0.001 is for perception-overload analysis.
# This is different from your earlier simple val plot with conf=0.25.
CONF_MIN = 0.001

IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
DET_K = "1,5,10,20,50"
MAX_DET = 1000


# -------------------------
# Helpers
# -------------------------
def must_exist(p, what="path"):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p


def prepare_eval_root(src_root, dst_root):
    """
    Converts flat YOLO layout:

        src_root/images
        src_root/labels

    into split layout:

        dst_root/images/test
        dst_root/labels/test
    """

    src_images = src_root / "images"
    src_labels = src_root / "labels"

    dst_images = dst_root / "images" / SPLIT
    dst_labels = dst_root / "labels" / SPLIT

    must_exist(src_images, f"{src_root}/images")
    must_exist(src_labels, f"{src_root}/labels")

    if dst_root.exists():
        shutil.rmtree(dst_root)

    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    image_exts = [".png", ".jpg", ".jpeg", ".webp", ".bmp",
                  ".PNG", ".JPG", ".JPEG", ".WEBP", ".BMP"]

    image_paths = []
    for ext in image_exts:
        image_paths.extend(src_images.glob(f"*{ext}"))

    image_paths = sorted(image_paths)

    print(f"Preparing {dst_root}")
    print(f"Found {len(image_paths)} images in {src_images}")

    if len(image_paths) == 0:
        raise RuntimeError(f"No images found in {src_images}")

    missing_labels = 0

    for img_path in image_paths:
        shutil.copy2(img_path, dst_images / img_path.name)

        label_path = src_labels / f"{img_path.stem}.txt"
        out_label_path = dst_labels / f"{img_path.stem}.txt"

        if label_path.exists():
            shutil.copy2(label_path, out_label_path)
        else:
            out_label_path.write_text("")
            missing_labels += 1

    print(f"Created: {dst_images}")
    print(f"Created: {dst_labels}")
    print(f"Missing labels filled as empty: {missing_labels}")
    print()


# -------------------------
# Checks
# -------------------------
must_exist(EVAL_SCRIPT, "evaluation_ab.py")
must_exist(MODEL_PATH, "model")
must_exist(CLEAN_FLAT_ROOT, "clean dataset root")
must_exist(PATCHED_FLAT_ROOT, "patched dataset root")

# -------------------------
# Prepare clean and patched roots in split format
# -------------------------
prepare_eval_root(CLEAN_FLAT_ROOT, CLEAN_EVAL_ROOT)
prepare_eval_root(PATCHED_FLAT_ROOT, PATCHED_EVAL_ROOT)

must_exist(CLEAN_EVAL_ROOT / "images" / SPLIT, f"clean images/{SPLIT}")
must_exist(CLEAN_EVAL_ROOT / "labels" / SPLIT, f"clean labels/{SPLIT}")
must_exist(PATCHED_EVAL_ROOT / "images" / SPLIT, f"patched images/{SPLIT}")
must_exist(PATCHED_EVAL_ROOT / "labels" / SPLIT, f"patched labels/{SPLIT}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------
# Run evaluation_ab.py
# -------------------------
cmd = [
    sys.executable, str(EVAL_SCRIPT),
    "--model", str(MODEL_PATH),
    "--clean_root", str(CLEAN_EVAL_ROOT),
    "--patched_root", str(PATCHED_EVAL_ROOT),
    "--out_root", str(OUT_ROOT),
    "--run_name", RUN_NAME,
    "--split", SPLIT,
    "--imgsz", str(IMGSZ),
    "--conf_min", str(CONF_MIN),
    "--iou", str(IOU),
    "--thresholds", THRESHOLDS,
    "--gt_iou_match", str(GT_IOU_MATCH),
    "--det_k", DET_K,
    "--max_det", str(MAX_DET),
]

print("[RUNNING]")
print(" ".join(cmd))

env = os.environ.copy()

# Important:
# Do NOT use this:
# env["CUDA_VISIBLE_DEVICES"] = ""
#
# That would force CPU. We leave CUDA visible.

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

full_log = []

for line in proc.stdout:
    print(line, end="")
    full_log.append(line)

proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"evaluation_ab.py failed with rc={proc.returncode}")

# -------------------------
# Read outputs
# -------------------------
run_dir = OUT_ROOT / RUN_NAME
combined_path = run_dir / "combined_metrics.json"
summary_path = run_dir / "summary.json"

must_exist(combined_path, "combined_metrics.json")
must_exist(summary_path, "summary.json")

summary = json.loads(summary_path.read_text())

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\nSaved:")
print("Run dir:", run_dir)
print("Combined:", combined_path)
print("Summary:", summary_path)

## 8. Detection visual comparisons

This saves side-by-side clean vs patched predictions for a few images.

In [ ]:
# =========================
# Validation / visualization prediction settings
# =========================
VAL_CONF = 0.25   # confidence threshold
VAL_IOU = 0.70    # NMS IoU threshold

In [ ]:
# =========================
# Visual clean vs patched predictions
# =========================
COMPARE_DIR = VIS_SAVE_DIR / 'clean_vs_realworld_rect_predictions'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

NUM_COMPARE = min(8, len(image_paths))
compare_paths = image_paths[:NUM_COMPARE]

for img_path in compare_paths:
    img_path = Path(img_path)
    patched_path = PATCH_DATASET_IMAGES / img_path.name

    clean_pred = model(str(img_path), imgsz=IMGSZ, conf=VAL_CONF, iou=VAL_IOU, verbose=False)[0]
    patched_pred = model(str(patched_path), imgsz=IMGSZ, conf=VAL_CONF, iou=VAL_IOU, verbose=False)[0]

    clean_plot = clean_pred.plot()
    patched_plot = patched_pred.plot()

    clean_pil = Image.fromarray(clean_plot[..., ::-1]) if clean_plot.shape[-1] == 3 else Image.fromarray(clean_plot)
    patched_pil = Image.fromarray(patched_plot[..., ::-1]) if patched_plot.shape[-1] == 3 else Image.fromarray(patched_plot)

    canvas = Image.new('RGB', (IMGSZ * 2, IMGSZ), 'white')
    canvas.paste(clean_pil.resize((IMGSZ, IMGSZ)), (0, 0))
    canvas.paste(patched_pil.resize((IMGSZ, IMGSZ)), (IMGSZ, 0))

    out_path = COMPARE_DIR / f'compare_{img_path.stem}.jpg'
    canvas.save(out_path, quality=95)

print('Saved comparisons to:', COMPARE_DIR)

# Display first few comparisons
compare_imgs = sorted(COMPARE_DIR.glob('compare_*.jpg'))[:4]
plt.figure(figsize=(14, 10))
for i, p in enumerate(compare_imgs):
    im = Image.open(p).convert('RGB')
    plt.subplot(len(compare_imgs), 1, i + 1)
    plt.imshow(im)
    plt.axis('off')
    plt.title(p.name + '   | left=clean, right=patched')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Save clean / patched images with and without detections
# =========================

from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np

# Safe defaults
VAL_CONF = globals().get("VAL_CONF", 0.25)
VAL_IOU = globals().get("VAL_IOU", 0.70)

EXPORT_DIR = Path(VIS_SAVE_DIR) / "clean_patch_with_without_detections"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

NUM_SAVE = min(8, len(image_paths))
save_paths = image_paths[:NUM_SAVE]

def pred_to_pil(result, size=(IMGSZ, IMGSZ)):
    """
    Convert YOLO result.plot() output to PIL image.
    """
    arr = result.plot()

    # Same convention you used earlier
    if arr.shape[-1] == 3:
        arr = arr[..., ::-1]

    return Image.fromarray(arr).convert("RGB").resize(size)

def add_title(img, title, title_h=35):
    """
    Add a small title above an image.
    """
    canvas = Image.new("RGB", (img.width, img.height + title_h), "white")
    canvas.paste(img, (0, title_h))

    draw = ImageDraw.Draw(canvas)
    draw.text((10, 10), title, fill="black")

    return canvas

saved_rows = []

for idx, img_path in enumerate(save_paths):
    img_path = Path(img_path)
    patched_path = PATCH_DATASET_IMAGES / img_path.name

    if not patched_path.exists():
        print(f"[SKIP] Missing patched image: {patched_path}")
        continue

    stem = img_path.stem

    # -------------------------
    # 1. Clean image, no detections
    # -------------------------
    clean_no_det = Image.open(img_path).convert("RGB").resize((IMGSZ, IMGSZ))

    # -------------------------
    # 2. Clean image, with detections
    # -------------------------
    clean_pred = model(
        str(img_path),
        imgsz=IMGSZ,
        conf=VAL_CONF,
        iou=VAL_IOU,
        verbose=False
    )[0]
    clean_with_det = pred_to_pil(clean_pred)

    # -------------------------
    # 3. Patched image, no detections
    # -------------------------
    patch_no_det = Image.open(patched_path).convert("RGB").resize((IMGSZ, IMGSZ))

    # -------------------------
    # 4. Patched image, with detections
    # -------------------------
    patch_pred = model(
        str(patched_path),
        imgsz=IMGSZ,
        conf=VAL_CONF,
        iou=VAL_IOU,
        verbose=False
    )[0]
    patch_with_det = pred_to_pil(patch_pred)

    # -------------------------
    # Save individual images
    # -------------------------
    clean_no_det_path = EXPORT_DIR / f"{stem}_01_clean_no_detections.jpg"
    clean_with_det_path = EXPORT_DIR / f"{stem}_02_clean_with_detections.jpg"
    patch_no_det_path = EXPORT_DIR / f"{stem}_03_patch_no_detections.jpg"
    patch_with_det_path = EXPORT_DIR / f"{stem}_04_patch_with_detections.jpg"

    clean_no_det.save(clean_no_det_path, quality=95)
    clean_with_det.save(clean_with_det_path, quality=95)
    patch_no_det.save(patch_no_det_path, quality=95)
    patch_with_det.save(patch_with_det_path, quality=95)

    # -------------------------
    # Save one comparison canvas
    # -------------------------
    panels = [
        add_title(clean_no_det, "Clean - no detections"),
        add_title(clean_with_det, "Clean - with detections"),
        add_title(patch_no_det, "Patch - no detections"),
        add_title(patch_with_det, "Patch - with detections"),
    ]

    canvas_w = IMGSZ * 4
    canvas_h = IMGSZ + 35
    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")

    for i, panel in enumerate(panels):
        canvas.paste(panel, (i * IMGSZ, 0))

    compare_path = EXPORT_DIR / f"{stem}_comparison_clean_patch_detections.jpg"
    canvas.save(compare_path, quality=95)

    saved_rows.append({
        "image": img_path.name,
        "clean_no_detections": str(clean_no_det_path),
        "clean_with_detections": str(clean_with_det_path),
        "patch_no_detections": str(patch_no_det_path),
        "patch_with_detections": str(patch_with_det_path),
        "comparison": str(compare_path),
    })

    print(f"[{idx}] Saved:", compare_path)

print("\nDone.")
print("Saved outputs to:", EXPORT_DIR)
print("Number of images processed:", len(saved_rows))

# Display first comparison
if len(saved_rows) > 0:
    first_compare = saved_rows[0]["comparison"]
    im = Image.open(first_compare).convert("RGB")
    plt.figure(figsize=(22, 6))
    plt.imshow(im)
    plt.axis("off")
    plt.title("Left to right: clean no det | clean det | patch no det | patch det")
    plt.show()

## 9. Notes

The saved patch to print/use as the adversarial texture is:

```text
realworld_rect_patch_training/patch_outputs/nova_patch_realworld_square.pt
realworld_rect_patch_training/patch_outputs/nova_patch_realworld_square.png
```

The rectangular visualization is only how the model sees it after the robot-style resize:

```text
realworld_rect_patch_training/patch_outputs/nova_patch_realworld_seen_rect.png
```

During training, the model never sees the square pasted directly. It only sees the real-world rectangle. That is the fix.